<a href="https://colab.research.google.com/github/xlphon/discourse_audit/blob/main/notebooks/01_colab_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# From Data Quality to Data Value: Colab Quickstart

This notebook runs the first two steps of the project:

1. Stream and sample public pretraining corpora from Hugging Face.
2. Extract transparent discourse/linguistic/governance proxy features.

Before running, upload this repository to GitHub and replace `YOUR_GITHUB_USERNAME` below.


In [ ]:
# EDIT THIS AFTER YOU UPLOAD THE REPO TO GITHUB.
GITHUB_USERNAME = "YOUR_G"
REPO_NAME = "llm-discourse-audit"
REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
print(REPO_URL)


## 1. Clone repo and install dependencies


In [ ]:
import os, shutil, pathlib

if pathlib.Path(REPO_NAME).exists():
    print(f"Repository folder {REPO_NAME} already exists.")
else:
    !git clone {REPO_URL}

%cd {REPO_NAME}
!pip -q install -r requirements.txt


## 2. Inspect and optionally edit dataset config

The default config samples 5,000 documents from each corpus:

- `fineweb`
- `fineweb_edu`

Increase `n_docs` only after the first run succeeds.


In [ ]:
from pathlib import Path
print(Path("configs/datasets.yaml").read_text())


Optional: edit `n_docs` from inside Colab.


In [ ]:
# Example: change 5000 to 20000 after the pipeline works.
# import pathlib
# p = pathlib.Path("configs/datasets.yaml")
# txt = p.read_text().replace("n_docs: 5000", "n_docs: 20000")
# p.write_text(txt)
# print(p.read_text())


## 3. Step 1: stream and sample datasets


In [ ]:
!python src/00_stream_sample.py \
  --only fineweb fineweb_edu \
  --config configs/datasets.yaml \
  --out_dir data/raw \
  --seed 42


Check sampled files.


In [ ]:
!ls -lh data/raw


## 4. Step 2: extract features


In [ ]:
!python src/01_extract_features.py \
  --raw_dir data/raw \
  --out data/features/features.parquet


## 5. Inspect output


In [ ]:
import pandas as pd

df = pd.read_parquet("data/features/features.parquet")
print(df.shape)
display(df.head())
display(df.groupby("corpus").size())


## 6. Quick feature comparison


In [ ]:
feature_cols = [
    "word_count",
    "type_token_ratio",
    "first_person_rate",
    "second_person_rate",
    "question_per_1k_words",
    "dialogue_like_line_rate",
    "imperative_proxy_line_rate",
    "boilerplate_count",
    "pii_risk",
    "code_flag",
]

summary = df.groupby("corpus")[feature_cols].mean().T
display(summary)


## 7. Save outputs to Google Drive, optional


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/llm-discourse-audit
# !cp -r data/features /content/drive/MyDrive/llm-discourse-audit/
# !cp -r data/raw /content/drive/MyDrive/llm-discourse-audit/
